# Notebook 07 - Evaluasi & SHAP Interpretability
**Proyek:** Pulsevera - Predict, Prevent, Prevail  
**Role:** AI Engineer (Fathan & Shafira)  
**Tujuan:** Bandingkan performa model ML vs DL, hitung metrik komprehensif (accuracy, precision, recall, F1, ROC-AUC), generate SHAP explanations untuk model terbaik, dan demonstrasi inferensi end-to-end dengan input user mentah.

**Output:**
- `ml-api/models/shap_explainer.pkl` — explainer untuk dipakai FastAPI
- Visualisasi: ROC, confusion matrix, SHAP summary, dependence plots

## 1. Setup

In [ ]:
import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score, roc_curve,
)

warnings.filterwarnings('ignore')

BASE_DIR = Path('..')
FINAL_DIR = BASE_DIR / 'data' / 'final'
MODELS_DIR = BASE_DIR / 'ml-api' / 'models'
FIGURES_DIR = BASE_DIR / 'notebooks' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print('SHAP', shap.__version__, '| TF', tf.__version__)

## 2. Load Data, Models & Metadata

In [ ]:
X_train = pd.read_csv(FINAL_DIR / 'X_train.csv')
X_test = pd.read_csv(FINAL_DIR / 'X_test.csv')
y_test = pd.read_csv(FINAL_DIR / 'y_test.csv').squeeze().values
FEATURE_ORDER = X_train.columns.tolist()

with open(MODELS_DIR / 'feature_metadata.json') as f:
    feature_meta = json.load(f)
with open(MODELS_DIR / 'dl_metadata.json') as f:
    dl_meta = json.load(f)

ml_model = joblib.load(MODELS_DIR / 'pulsevera_ml_model.pkl')
scaler = joblib.load(MODELS_DIR / 'scaler.pkl')

@keras.saving.register_keras_serializable(package='pulsevera', name='focal_loss')
def focal_loss(y_true, y_pred, gamma=2.0, alpha=0.25):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
    bce = -(y_true * tf.math.log(y_pred) + (1 - y_true) * tf.math.log(1 - y_pred))
    p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
    alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
    return tf.reduce_mean(alpha_t * tf.pow(1.0 - p_t, gamma) * bce)

dl_model = keras.models.load_model(
    MODELS_DIR / 'pulsevera_dl_model.keras',
    custom_objects={'focal_loss': focal_loss}, safe_mode=False,
)

print(f'Best ML: {feature_meta["best_model_name"]}  (scaling={feature_meta["needs_scaling"]})')
print(f'DL best threshold: {dl_meta["best_threshold"]}')

## 3. Evaluasi Komprehensif: ML vs DL

In [ ]:
def predict_proba_ml(X):
    X_in = scaler.transform(X) if feature_meta['needs_scaling'] else X
    return ml_model.predict_proba(X_in)[:, 1]

proba_ml = predict_proba_ml(X_test)
proba_dl = dl_model.predict(scaler.transform(X_test).astype('float32'), batch_size=1024, verbose=0).ravel()

pred_ml = (proba_ml >= 0.5).astype(int)
pred_dl = (proba_dl >= dl_meta['best_threshold']).astype(int)

def metric_row(name, y_true, y_pred, y_proba):
    return {
        'model': name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_pos': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'recall_pos': recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        'f1_pos': f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_proba),
    }

comparison = pd.DataFrame([
    metric_row(feature_meta['best_model_name'], y_test, pred_ml, proba_ml),
    metric_row('Deep Learning', y_test, pred_dl, proba_dl),
])
print(comparison.round(4).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

fpr_ml, tpr_ml, _ = roc_curve(y_test, proba_ml)
fpr_dl, tpr_dl, _ = roc_curve(y_test, proba_dl)
axes[0].plot(fpr_ml, tpr_ml, label=f'{feature_meta["best_model_name"]} (AUC={roc_auc_score(y_test, proba_ml):.3f})', linewidth=2)
axes[0].plot(fpr_dl, tpr_dl, label=f'Deep Learning (AUC={roc_auc_score(y_test, proba_dl):.3f})', linewidth=2)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].set_title('ROC Curves'); axes[0].legend()

comparison.set_index('model').plot(kind='bar', ax=axes[1], edgecolor='white', width=0.85)
axes[1].set_ylim(0, 1); axes[1].set_ylabel('Score'); axes[1].set_title('Perbandingan Metrik')
axes[1].axhline(0.85, color='gray', linestyle='--', alpha=0.5)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'final_model_comparison.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, name, pred in [(axes[0], feature_meta['best_model_name'], pred_ml), (axes[1], 'Deep Learning', pred_dl)]:
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
    ax.set_title(f'{name}', fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'final_confusion_matrices.png', dpi=150)
plt.show()

## 4. SHAP — Interpretability untuk Model ML

Strategi: `TreeExplainer` (sangat cepat, eksak) bila model tree-based; fallback `LinearExplainer` / `KernelExplainer` untuk Logistic Regression.

Gunakan sample 2000 baris untuk plot global (cukup representatif, hemat memori).

In [ ]:
SHAP_SAMPLE = 2000
rng = np.random.RandomState(42)
sample_idx = rng.choice(len(X_test), size=min(SHAP_SAMPLE, len(X_test)), replace=False)
X_shap = X_test.iloc[sample_idx].copy()

if hasattr(ml_model, 'estimators_'):
    explainer = shap.TreeExplainer(ml_model)
    X_shap_in = X_shap
elif hasattr(ml_model, 'coef_'):
    X_train_sample = X_train.sample(500, random_state=42)
    X_train_in = scaler.transform(X_train_sample) if feature_meta['needs_scaling'] else X_train_sample.values
    explainer = shap.LinearExplainer(ml_model, X_train_in, feature_names=FEATURE_ORDER)
    X_shap_in = scaler.transform(X_shap) if feature_meta['needs_scaling'] else X_shap.values
else:
    background = shap.sample(X_train, 100, random_state=42)
    explainer = shap.KernelExplainer(lambda d: ml_model.predict_proba(d)[:, 1], background)
    X_shap_in = X_shap

print('Hitung SHAP values ...')
shap_values = explainer.shap_values(X_shap_in)
if isinstance(shap_values, list):
    shap_values_pos = shap_values[1] if len(shap_values) > 1 else shap_values[0]
else:
    shap_values_pos = shap_values
    if shap_values_pos.ndim == 3:
        shap_values_pos = shap_values_pos[:, :, 1]
print('SHAP shape:', np.array(shap_values_pos).shape)

In [ ]:
plt.figure(figsize=(9, 7))
shap.summary_plot(shap_values_pos, X_shap, feature_names=FEATURE_ORDER, show=False, plot_size=None)
plt.title('SHAP Summary — Kontribusi Fitur terhadap Risiko Serangan Jantung', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))
shap.summary_plot(shap_values_pos, X_shap, feature_names=FEATURE_ORDER, plot_type='bar', show=False)
plt.title('Mean |SHAP value| — Importance Global', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'shap_importance_bar.png', dpi=150, bbox_inches='tight')
plt.show()

global_importance = pd.Series(np.abs(shap_values_pos).mean(axis=0), index=FEATURE_ORDER).sort_values(ascending=False)
print('Top 10 fitur paling berpengaruh:')
print(global_importance.head(10))

## 5. SHAP Local — Explain Satu Prediksi

In [ ]:
positive_indices = np.where(y_test[sample_idx] == 1)[0]
demo_idx = positive_indices[0] if len(positive_indices) else 0
demo_row = X_shap.iloc[[demo_idx]]
demo_shap = shap_values_pos[demo_idx]

print('Input pasien:')
print(demo_row.T.iloc[:15])
print(f'\nProba prediksi: {predict_proba_ml(demo_row)[0]:.4f}')
print(f'Label sebenarnya: {y_test[sample_idx][demo_idx]}')

top_factors = pd.Series(demo_shap, index=FEATURE_ORDER).abs().nlargest(5)
print(f'\nTop 5 faktor risiko (per pasien):')
for feat, contrib in top_factors.items():
    direction = 'menaikkan' if pd.Series(demo_shap, index=FEATURE_ORDER)[feat] > 0 else 'menurunkan'
    print(f'  {feat:30s}: |SHAP|={contrib:.4f}  ({direction} risiko)')

In [ ]:
plt.figure()
expected_value = explainer.expected_value
if isinstance(expected_value, (list, np.ndarray)):
    arr = np.array(expected_value).flatten()
    expected_value = float(arr[1] if arr.size > 1 else arr[0])

shap.force_plot(
    expected_value, demo_shap, demo_row.iloc[0],
    feature_names=FEATURE_ORDER, matplotlib=True, show=False,
)
plt.title('SHAP Force — kontribusi fitur untuk pasien ini', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'shap_force_demo.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Simpan SHAP Explainer untuk FastAPI

In [ ]:
explainer_path = MODELS_DIR / 'shap_explainer.pkl'
try:
    joblib.dump(explainer, explainer_path)
    print(f'Explainer disimpan: {explainer_path} ({explainer_path.stat().st_size / 1024**2:.2f} MB)')
except Exception as exc:
    print(f'Tidak bisa serialize explainer ({type(exc).__name__}); FastAPI akan generate ulang on-the-fly.')

shap_meta = {
    'explainer_type': type(explainer).__name__,
    'sample_size': int(len(X_shap)),
    'global_importance_top10': global_importance.head(10).round(4).to_dict(),
}
with open(MODELS_DIR / 'shap_metadata.json', 'w') as f:
    json.dump(shap_meta, f, indent=2)
print(f'SHAP metadata: {MODELS_DIR / "shap_metadata.json"}')

## 7. Demo Inferensi End-to-End (Simulasi Request Web Form)

Gunakan `ml-api/preprocessing.py` agar pipeline identik dengan FastAPI.

In [ ]:
import sys
sys.path.insert(0, str(BASE_DIR / 'ml-api'))
from preprocessing import preprocess_user_input  # noqa: E402
from inference import generate_recommendations  # noqa: E402

user_inputs = [
    {
        'sex': 'Male', 'age_category': 11, 'height_meters': 1.70, 'weight_kg': 95,
        'sleep_hours': 5, 'physical_activities': 'No', 'smoker_status': 'Current-every',
        'alcohol': 'Yes', 'general_health': 'Fair', 'diabetes': 'Yes',
    },
    {
        'sex': 'Female', 'age_category': 3, 'height_meters': 1.65, 'weight_kg': 58,
        'sleep_hours': 8, 'physical_activities': 'Yes', 'smoker_status': 'Never',
        'alcohol': 'No', 'general_health': 'Very good', 'diabetes': 'No',
    },
]

for i, ui in enumerate(user_inputs, 1):
    X_in = preprocess_user_input(ui)
    proba = predict_proba_ml(X_in)[0]
    label = 'Tinggi' if proba >= 0.6 else 'Sedang' if proba >= 0.3 else 'Rendah'
    sh = explainer.shap_values(scaler.transform(X_in) if feature_meta['needs_scaling'] else X_in)
    if isinstance(sh, list):
        sh = sh[1] if len(sh) > 1 else sh[0]
    if np.array(sh).ndim == 3:
        sh = sh[:, :, 1]
    top3 = pd.Series(np.array(sh)[0], index=FEATURE_ORDER).abs().nlargest(3).index.tolist()
    recs = generate_recommendations(ui, top3)
    print(f'=== Pasien {i} ===')
    print(f'  Risk score      : {proba:.4f} ({label})')
    print(f'  Top risk factors: {top3}')
    print(f'  Rekomendasi     : {recs}\n')

## 8. Final Checklist AI Engineer

| Item | Status |
|---|---|
| 3 ML models (LR, RF, DT) | ✓ |
| Class imbalance (SMOTE + class_weight) | ✓ |
| Hyperparameter tuning (RandomizedSearchCV) | ✓ |
| Deep Learning + Functional API | ✓ |
| Custom component (focal_loss + EarlyStoppingByRecall) | ✓ |
| Evaluasi accuracy / precision / recall / F1 / ROC-AUC | ✓ |
| SHAP interpretability | ✓ |
| Inference code | ✓ |
| Model format `.keras` + `.pkl` | ✓ |
| TensorBoard logs | ✓ |

Lanjut: `ml-api/main.py` sudah siap dipakai FastAPI untuk Full-Stack.